## 5 Importing Standard Earth Science Datasets
## 5.1 Text

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pygrib
import xarray as xr
from netCDF4 import Dataset

In [ ]:
filename = "20250101_20250131_CalTech.lev20"
aeronet_file = Path("./data/txt") / filename
aeronet = pd.read_csv(aeronet_file, skiprows=6)

In [ ]:
aeronet.head()

In [ ]:
list(aeronet.columns)

In [ ]:
aeronet["AOD_500nm"]

In [ ]:
drops = list(aeronet.columns[20:])
print(drops)

In [ ]:
aeronet_short = aeronet.drop(columns=drops)

In [ ]:
header = None
data_rows = []

with open(aeronet_file) as data:
    for line_num, row in enumerate(data):
        if line_num < 6:  # Skip first 6 rows
            continue
        elif line_num == 6:  # Row 7 is the header
            header = row.strip().split(",")
        else:  # Data rows
            values = row.strip().split(",")
            data_rows.append(values)

# Convert to pandas DataFrame
aeronet_manual = pd.DataFrame(data_rows, columns=header)
aeronet_manual.head()

## 5.2 NetCDF

In [ ]:
filename = "JRR-AOD_v3r2_j01_s202306071807538_e202306071809183_c202306071842528.nc"
aod_file = Path("./data/aod") / filename
aod_nc = Dataset(aod_file)

In [ ]:
print(aod_nc)

In [ ]:
aod_nc.variables.keys()

In [ ]:
AOD_550_nc = aod_nc.variables["AOD550"]
type(AOD_550_nc)

In [ ]:
AOD_550 = np.array(AOD_550_nc[:,:], copy=True)
AOD_550.shape, type(AOD_550)

In [ ]:
AOD_550

In [ ]:
avg_AOD = AOD_550.mean()
print(avg_AOD)

In [ ]:
print(aod_nc.variables["AOD550"])

### 5.2.1. Manually Creating a Mask Variable Using True and False Values

In [ ]:
AOD_MISSING = aod_nc.variables["AOD550"]._FillValue
AOD_MISSING

In [ ]:
keep_rows = AOD_550 != AOD_MISSING
AOD_550[50:60, 100], keep_rows[50:60, 100]

In [ ]:
AOD_550_filtered = AOD_550[keep_rows]
AOD_550_filtered

In [ ]:
avg_AOD_filtered = AOD_550_filtered.mean()
avg_AOD_filtered

In [ ]:
AOD_550.size, AOD_550_filtered.size

### 5.2.2. Using NumPy Masked Arrays to Filter Automatically

In [ ]:
AOD_550_ma = aod_nc.variables["AOD550"][:,:]
type(AOD_550_ma)

In [ ]:
AOD_550_ma[50:60, 100]

In [ ]:
avg_AOD_ma = AOD_550_ma.mean()
avg_AOD_ma

In [ ]:
aod_nc.variables["AOD550"].valid_range

## 5.3 HDF

In [ ]:
filename = "3B-HHR.MS.MRG.3IMERG.20240927-S073000-E075959.0450.V07B.HDF5"
imerg_file = Path("./data/imerg") / filename
imerg_h5 = h5py.File(imerg_file, "r")
imerg_h5

In [ ]:
list(imerg_h5)

In [ ]:
print(list(imerg_h5["Grid"].keys()))

In [ ]:
imerg_h5.visit(print)

In [ ]:
precip = imerg_h5["Grid/precipitation"][:,:,:]
precip

In [ ]:
print(list(imerg_h5["Grid/precipitation"].attrs))

In [ ]:
PRECIP_MISSING = imerg_h5["Grid/precipitation"].attrs["_FillValue"]
PRECIP_MISSING

In [ ]:
precip_mask = (precip == PRECIP_MISSING)
precip_filtered = np.ma.masked_array(precip, mask=precip_mask)
precip_filtered

In [ ]:
precip_filtered.mean()

## 5.4 GRIB2

In [ ]:
filename = "gfs.t00z.pgrb2.1p00.anl"
gfs_file = Path("./data/gfs") / filename
gfs_grb2 = pygrib.open(gfs_file)
records = [str(grb) for grb in gfs_grb2]

In [ ]:
records[511]

In [ ]:
gfs_temps = gfs_grb2.select(name="Temperature")
gfs_temps

In [ ]:
gfs_temp = gfs_grb2[437]
gfs_temp

In [ ]:
gfs_temp.values

In [ ]:
gfs_temp.keys()

In [ ]:
gfs_lat = gfs_temp.latitudes
gfs_lon = gfs_temp.longitudes
gfs_level = gfs_temp.level
gfs_units = gfs_temp.units
gfs_analysis_date = gfs_temp.analDate
gfs_fcst_time = gfs_temp.forecastTime

## 5.5 Importing and Exploring Data Using Xarray
### 5.5.1 Opening netCDF files

In [ ]:
filename = "NUCAPS-EDR_v3r2_j01_s202501092048589_e202501092049287_c202501092139200.nc"
nucaps_file = Path("data/nucaps") / filename
nucaps = xr.open_dataset(nucaps_file, engine="h5netcdf")
nucaps

### 5.5.2. Examining Vertical Cross Sections

In [ ]:
nucaps_profile = nucaps["Temperature"].sel(Number_of_CrIS_FORs=103)
nucaps_profile

In [ ]:
nucaps_lat = nucaps_profile["Latitude"].item()
nucaps_lon = nucaps_profile["Longitude"].item()
print(nucaps_lat, nucaps_lon)

In [ ]:
np.set_printoptions(suppress=True, precision=2)
nucaps_profile["Pressure"].values

In [ ]:
target_lat = 34.1
target_lon = -118.1

In [ ]:
lat_data = nucaps["Latitude"].values
lon_data = nucaps["Longitude"].values

In [ ]:
lat_diff = np.abs(lat_data - target_lat)
lon_diff = np.abs(lon_data - target_lon)
distance = np.sqrt(lat_diff**2 + lon_diff**2)
distance 

In [ ]:
nearest_for_idx = np.argmin(distance)
nearest_for_idx

In [ ]:
nucaps_temp_caltech = nucaps.isel(Number_of_CrIS_FORs=nearest_for_idx)

### 5.5.3. Examining Horizontal Cross Sections

In [ ]:
nucaps_gradient_idx = nucaps.sel(Number_of_P_Levels = 74)

In [ ]:
target_pressure = 500
nucaps_pressure = nucaps["Pressure"].values
pressure_diff = np.abs(nucaps_pressure - target_pressure)
pressure_diff

In [ ]:
nearest_p_indices = np.nanargmin(pressure_diff, axis=1)

In [ ]:
p_indices  = xr.DataArray(nearest_p_indices, dims="Number_of_CrIS_FORs")

In [ ]:
nucaps_gradient = nucaps.isel(Number_of_P_Levels=p_indices)
nucaps_gradient["Temperature"].values

### 5.2.2 Opening GRIB2

In [ ]:
filter_keys = {"filter_by_keys": {"typeOfLevel": "isobaricInhPa", "name": "Temperature"}}

In [ ]:
gfs_xr = xr.open_dataset(gfs_file, engine="cfgrib", backend_kwargs=filter_keys)
gfs_xr

In [ ]:
temp_850  =  gfs_xr.sel(isobaricInhPa=850)
temp_850

In [ ]:
temp_900 = gfs_xr.sel(isobaricInhPa=918, method="nearest")
temp_900["isobaricInhPa"].values

### 5.2.3 Accessing datasets using OpenDAP

In [ ]:
base_url = "http://psl.noaa.gov/thredds/dodsC/Datasets/"
product_url = "noaa.ersst.v6/sst.mnmean.nc"
ersst = xr.open_dataset(base_url+product_url)
ersst